### Installation

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps tokenizers trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
# causal_conv1d is supported only on torch==2.8.0. If you have newer torch versions, please wait 10 minutes!
!uv pip install --no-build-isolation flash-linear-attention causal_conv1d==1.6.0

In [ ]:
!pip install wandb -qU

In [ ]:
import wandb
wandb.login()

In [ ]:
import os
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

In [ ]:
# 1. Dọn dẹp bộ nhớ CUDA trước khi nạp lại
import torch
import gc

def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()

clear_gpu()

# 2. Khởi tạo mô hình với 4-bit quantization để tránh OOM trên T4
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "phuclhp1922/qwen3.5_0.8B_translation_merged_16bit",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True, # BẬT 4-bit quantization
    device_map = "cuda",
    use_gradient_checkpointing = "unsloth",
)

### Unsloth

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    finetune_language_layers   = True, # False if not finetuning language layers
    finetune_attention_modules = True, # False if not finetuning attention layers
    finetune_mlp_modules       = True, # False if not finetuning MLP layers

    r = 8,           # The larger, the higher the accuracy, but might overfit
    lora_alpha = 8,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
    # target_modules = "all-linear", # Optional now! Can specify a list if needed
)

### Load your CSV data

First, make sure you have uploaded your CSV file to your Colab environment (e.g., by dragging and dropping it into the 'Files' tab on the left sidebar). Then, we can load it into a pandas DataFrame.

In [ ]:
import pandas as pd
import glob

# List of your JSONL files
jsonl_files = glob.glob('/content/*.jsonl')

# Load and concatenate all JSONL files
df_list = [pd.read_json(f, lines=True) for f in jsonl_files]
df = pd.concat(df_list, ignore_index=True)

# Display the first 5 rows to verify loading
print(f"Loaded {len(df)} rows from {len(jsonl_files)} files.")
display(df.head())

In [ ]:
df.iloc[1]['messages']

### Updating Training Arguments for WandB
I will now update the trainer configuration to report to Weights & Biases.

In [ ]:
# Khởi tạo project WandB mới
wandb.init(
    project="bct-qwen-translation-vn",
    config={
        "learning_rate": 4e-5,
        "architecture": "Qwen3.5-4B",
        "dataset": "translation-en-vi",
        "epochs": 1,
    }
)

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
from datasets import Dataset
import torch
import gc
from trl import SFTTrainer, SFTConfig
from transformers import DataCollatorForLanguageModeling
import numpy as np
import copy

# 1. Dọn dẹp bộ nhớ trước khi xử lý
gc.collect()
torch.cuda.empty_cache()

# 2. Xử lý dữ liệu trực tiếp từ message payloads
all_input_ids = []
all_attention_mask = []

for _, row in df.iterrows():
    raw_messages = copy.deepcopy(row['messages'])
    formatted_messages = []

    for msg in raw_messages:
        role = msg.get("role")
        content = msg.get("content")

        if isinstance(content, list):
            text_parts = [item["text"] for item in content if isinstance(item, dict) and "text" in item]
            text_content = " ".join(text_parts)
        else:
            text_content = str(content) if content is not None else ""

        formatted_messages.append({
            "role": role,
            "content": [{"type": "text", "text": text_content}]
        })

    outputs = tokenizer.apply_chat_template(
        formatted_messages,
        tokenize = True,
        add_generation_prompt = False,
        truncation = True,
        max_length = 2048,
        return_dict = True,
    )

    ids = outputs["input_ids"]
    mask = outputs["attention_mask"]

    if isinstance(ids[0], (list, np.ndarray, torch.Tensor)):
        ids = ids[0]
        mask = mask[0]

    all_input_ids.append(ids)
    all_attention_mask.append(mask)

# 3. Tạo Dataset
train_ds = Dataset.from_dict({
    "input_ids": all_input_ids,
    "attention_mask": all_attention_mask,
})

# 4. Sử dụng DataCollator chuẩn
data_collator = DataCollatorForLanguageModeling(tokenizer = tokenizer, mlm = False)

# 5. Khởi tạo Trainer (Giảm batch size để tránh OOM)
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    dataset_text_field = None,
    data_collator = data_collator,
    max_seq_length = 2048,
    args = SFTConfig(
        per_device_train_batch_size = 2, # Giảm xuống 1 để tiết kiệm VRAM
        gradient_accumulation_steps = 4, # Tăng accumulation để giữ hiệu quả batch size = 8
        warmup_steps = 50,
        num_train_epochs = 2,
        max_steps = -1,
        learning_rate = 4e-5,
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs",
        report_to = "wandb",
    ),
)

# 6. Chạy training
trainer_stats = trainer.train()

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")

In [ ]:
# @title Inference example

FastLanguageModel.for_inference(model) # Enable for inference!

instruction = "Translate this English sentence to Vietnamese:"
prompt = f"/no_think {instruction} Hello world."

messages = [
    {"role": "user", "content": prompt}
]

# Sử dụng chat template
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt = True, tokenize = False)

# SỬA LỖI: Đảm bảo tokenizer không cố gắng xử lý hình ảnh
inputs = tokenizer(
    text = [input_text], # Truyền vào dạng list text
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)

# Sinh văn bản
_ = model.generate(
    input_ids = inputs.input_ids,
    attention_mask = inputs.attention_mask,
    streamer = text_streamer,
    max_new_tokens = 128,
    use_cache = True,
    temperature = 1.5,
    min_p = 0.1
)

<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
model.save_pretrained("qwen_lora")  # Local saving
tokenizer.save_pretrained("qwen_lora")
model.push_to_hub("phuclhp1922/bct_qwen3.5_0.8B_lora", token = os.environ['HF_TOKEN']) # Online saving
tokenizer.push_to_hub("phuclhp1922/bct_qwen3.5_0.8B_lora", token = os.environ['HF_TOKEN']) # Online saving

### Exporting to Float16 for Production
This cell will merge your LoRA adapters with the base model and upload the result to Hugging Face. This version is ideal for vLLM or other inference engines.

In [ ]:
# Merge to 16bit and push to Hugging Face
model.push_to_hub_merged(
    "phuclhp1922/bct_qwen3.5_0.8B_translation_merged_16bit",
    tokenizer,
    save_method = "merged_16bit",
    token = os.environ['HF_TOKEN']
)

### Final GGUF Export
This cell converts your fine-tuned model into GGUF format for use in Ollama or llama.cpp. We use `q8_0` for high precision or `q4_k_m` for better compression.

In [ ]:
# Push GGUF version to Hugging Face
model.push_to_hub_gguf(
    "phuclhp1922/bct_qwen3.5_0.8B_translation_gguf",
    tokenizer,
    quantization_method = ["q8_0"],
    token = os.environ['HF_TOKEN']
)